In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from src.taxi_lts.run_analysis import load_protected_split
from src.taxi_lts.explore_training import PREDICTORS, TARGET
from src.taxi_lts.stress_test import fit_pair, contaminate_training_response
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

root = Path.cwd()
train, test = load_protected_split(root / "data/processed/taxi_modeling_50000.csv")
flow = pd.read_csv(root / "outputs/tables/data_quality_flow.csv")
display(flow.iloc[:-1])
print("Training", len(train), "Test", len(test))

/Users/ak007/Projects/Regression_Techniques_Project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,stage,records_before,records_removed,records_after
0,original_records,3724889,0,3724889
1,pickup_in_january_2026,3724889,7,3724882
2,standard_rate_code_1,3724882,1334391,2390491
3,positive_trip_distance,2390491,17940,2372551
4,positive_fare_amount,2372551,33753,2338798
5,duration_at_least_1_minute,2338798,47068,2291730
6,passenger_count_at_least_1,2291730,13648,2278082
7,eligible_records,2278082,0,2278082


Training 40000 Test 10000


In [2]:
scaler = StandardScaler().fit(train[PREDICTORS])
X = scaler.transform(train[PREDICTORS])
y = train[TARGET].to_numpy(dtype=float)
with threadpool_limits(limits=2):
    clean_ols, clean_lts = fit_pair(X, y)
print("Raw LTS h:", len(clean_lts.raw_estimator.best_h_subset))
print("MAD scale:", round(clean_lts.raw_robust_scale, 6))
print("Retained for reweighting:", clean_lts.reweighting_weights.sum())

Raw LTS h: 30000
MAD scale: 0.510606
Retained for reweighting: 33548


In [3]:
from robpy.regression import FastLTSRegression
from scipy.stats import median_abs_deviation, chi2
from sklearn.linear_model import LinearRegression
import logging

with threadpool_limits(limits=2):
    raw = FastLTSRegression(
        alpha=0.75, n_initial_subsets=500, reweighting=False, random_state=42
    )
    raw.fit(X, y, verbosity=logging.WARNING)
r = y - raw.predict(X).reshape(-1)
scale = median_abs_deviation(r, scale="normal")
cutoff = np.sqrt(chi2.ppf(0.975, df=1))
weights = np.abs(r / scale) <= cutoff
reweighted = LinearRegression().fit(X[weights], y[weights])
assert np.allclose(reweighted.coef_, clean_lts.reweighted_model.coef_)
print("h", len(raw.best_h_subset))
print("Scale", round(scale, 6), "Weight 1", int(weights.sum()))

h 30000
Scale 0.510606 Weight 1 33548


In [4]:
from src.taxi_lts.explore_training import fit_diagnostic_ols, coefficient_table

diagnostic_ols, hc3 = fit_diagnostic_ols(train)
display(coefficient_table(diagnostic_ols, hc3).round(4))

,term,coefficient,standard_error_conventional,standard_error_hc3,hc3_ci_lower_95,hc3_ci_upper_95,hc3_p_value
0,const,5.2406,0.0407,0.2853,4.6815,5.7998,0.0000
1,trip_distance,3.6218,0.0059,0.0958,3.4339,3.8096,0.0000
2,duration_min,0.0824,0.0010,0.0392,0.0055,0.1593,0.0357
3,passenger_count,0.1380,0.0246,0.0360,0.0674,0.2086,0.0001
4,pickup_hour_sin,-0.5781,0.0270,0.0611,-0.6979,-0.4583,0.0000
5,pickup_hour_cos,-1.0891,0.0253,0.0876,-1.2609,-0.9173,0.0000
6,is_weekend,-0.7448,0.0376,0.0653,-0.8728,-0.6169,0.0000


In [5]:
from src.taxi_lts.run_analysis import prediction_metrics

Xt = scaler.transform(test[PREDICTORS])
yt = test[TARGET].to_numpy(dtype=float)
rows = []
for name, model in [
    ("OLS", clean_ols),
    ("Reweighted LTS", clean_lts.reweighted_model),
]:
    rows.append({"Estimator": name, **prediction_metrics(yt, model.predict(Xt))})
display(pd.DataFrame(rows).round(6))

,Estimator,mae,rmse,median_absolute_error,r_squared
0,OLS,2.086049,3.253342,1.594237,0.923439
1,Reweighted LTS,1.015238,8.756125,0.346920,0.445409


In [6]:
ys, known = contaminate_training_response(train)
assert known.sum() == 2000
assert np.allclose(ys[known] - y[known], 100)
assert np.array_equal(ys[~known], y[~known])
with threadpool_limits(limits=2):
    stressed_ols, stressed_lts = fit_pair(X, ys)
rows = []
for name, model in [
    ("OLS", stressed_ols),
    ("Reweighted LTS", stressed_lts.reweighted_model),
]:
    rows.append({"Estimator": name, **prediction_metrics(yt, model.predict(Xt))})
display(pd.DataFrame(rows).round(6))

,Estimator,mae,rmse,median_absolute_error,r_squared
0,OLS,5.382720,5.939032,5.578209,0.744859
1,Reweighted LTS,1.000121,8.737124,0.347235,0.447813


In [7]:
flag = (~stressed_lts.reweighting_weights) | (
    np.abs(stressed_lts.raw_standardized_residuals) > 2.5
)
tp = int(np.sum(flag & known))
fp = int(np.sum(flag & ~known))
fn = int(np.sum(~flag & known))
print("TP", tp, "FP", fp, "FN", fn)
print("Recall", tp / known.sum())
print("Precision", round(tp / (tp + fp), 6))

TP 2000 FP 5684 FN 0
Recall 1.0
Precision 0.260281
